### Performance evaluations: Precision - Recall and Average Percision ###

In [8]:
import sys
import os
import numpy as np
import json
import copy
import glob
import pandas as pd
import logging
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from sklearn import metrics
import seaborn as sns

logger = logging.getLogger(__name__)

# PyTorch
import torch
from torchvision import ops

%load_ext autoreload
%autoreload 2
import computervision
from computervision.imageproc import is_image, ImageData, clipxywh, xyxy2xywh, xywh2xyxy, plot_boxes
from computervision.datasets import DETRdataset
from computervision.transformations import AugmentationTransform
from computervision.performance import DetectionMetrics
from computervision.inference import DETRinference, get_gpu_info

print(f'Project version: {computervision.__version__}')
print(f'Authors: {computervision.__authors__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project version: v0.0.2
Authors: The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [3]:
# Check GPU availability
device, device_str = get_gpu_info()

CUDA available: True
Number of GPUs found:  1
Current device ID: 0
GPU device name:   NVIDIA GeForce RTX 3060 Laptop GPU
PyTorch version:   2.8.0a0+34c6371d24.nv25.08
CUDA version:      13.0
CUDNN version:     91200
Current device:    cuda:0


### Test data ###

In [4]:
# Dentex test data
data_dir = os.environ.get('DATA')
dataset_name = 'dataset_object_dentex_251011'
image_dir = os.path.join(data_dir, dataset_name, 'test')
test_df_file_name = 'dataset_object_dentex_251011_test.parquet'
test_df_file = os.path.join(image_dir, test_df_file_name)
df = pd.read_parquet(test_df_file)

# Filter the data frame
dset_col = 'dset'
pos_col = 'ada'
file_col = 'file_name'
bbox_col = 'bbox'
df = df.loc[(df[dset_col] == 'test') & (df['transformation'] == 5)]
display(df.head(2))
print(f'Images in test data: {len(df[file_col].unique())}')
print(f'Annotations:         {df.shape[0]}')

,bbox,quadrant,ada,file_name,file_base_name,quadrants,height,width,transformation,transformation_name,dset
10256,"[572, 228, 67, 338]",1,8,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test
10257,"[511, 234, 91, 332]",1,7,test_train_219_01_05.png,train_219,1,640,640,5,test_set,test


Images in test data: 160
Annotations:         1275


### Model ###

In [26]:
model_name = 'rtdetr_dtx_251012_08'
model_dir = os.path.join(data_dir, 'model', model_name)

checkpoint_paths = glob.glob(os.path.join(model_dir, 'checkpoint-*'))
checkpoint_numbers = [int(os.path.basename(checkpoint).split('-')[-1]) for checkpoint in checkpoint_list]
checkpoints = dict(zip(checkpoint_numbers, checkpoint_paths))

# checkpoint_dir = os.path.join(model_dir, f'checkpoint-{checkpoint}')
display(checkpoints)

model_config_file = os.path.join(model_dir, f'{model_name}.json')
with open(model_config_file, mode='r') as file:
    model_config = json.load(file)
display(*list(model_config.keys()), sep='\n')

# Image processor
processor = DETRinference(device_name='cuda:0', 
                          checkpoint_path=checkpoint_paths[0]).processor

{19600: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-19600',
 119900: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119900',
 120000: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-120000',
 119950: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119950',
 119850: '/app/data_model/model/rtdetr_dtx_251012_08/checkpoint-119850'}

'model_info'

'id2label'

'training_args'

'processor_params'

'bbox_format'

### Predict bounding boxes on the test data ###

In [28]:
# Create a PyTorch data set
transforms = AugmentationTransform().get_transforms(name='val')
bbox_format = model_config.get('bbox_format')
dataset = DETRdataset(data=df.copy(), 
                      image_processor=processor, 
                      image_dir=image_dir, 
                      file_name_col=file_col, 
                      label_id_col=None, 
                      bbox_col=None, 
                      bbox_format=bbox_format, 
                      transforms=transforms)
print(f'Total images in test data: {len(dataset)}')

Total images in test data: 160


In [35]:
# Run the forward pass to get the predictions for all checkpoints
threshold = 0.05
pred_raw_list = []
for c, (checkpoint, checkpoint_path) in enumerate(checkpoints.items()):
    print(f'Running checkpoint {checkpoint} {c + 1} / {len(checkpoints)}')
    dtr = DETRinference(device_name='cuda:0', 
                        checkpoint_path=checkpoint_path,
                        batch_size=16)
    pred = dtr.predict_on_dataset(dataset, threshold=threshold)
    pred = pred.assign(checkpoint=checkpoint, pred_threshold=threshold)
    pred_raw_list.append(pred)
pred_raw = pd.concat(pred_raw_list, axis=0, ignore_index=True).\
                sort_values(by='checkpoint', ascending=True).\
                reset_index(drop=True)

Running checkpoint 19600 1 / 5
Predicting batch 10 of 10.
Running checkpoint 119900 2 / 5
Predicting batch 10 of 10.
Running checkpoint 120000 3 / 5
Predicting batch 10 of 10.
Running checkpoint 119950 4 / 5
Predicting batch 10 of 10.
Running checkpoint 119850 5 / 5
Predicting batch 10 of 10.


In [36]:
# Make a copy of the predictions 
pred = copy.deepcopy(pred_raw)

# Add the file names to the data frame
file_names = df[file_col].unique()
id2file = dict(zip(range(len(file_names)), file_names))
pred[file_col] = pred['image_id'].apply(lambda image_id: id2file.get(image_id))

# Add the label names (the positions) to the data frame
id2label = {int(category_id): int(label) for category_id, label in model_config.get('id2label').items()}
pred[pos_col] = pred['category_id'].apply(lambda category_id: id2label.get(category_id))

print(f'Images in output data:     {len(pred['image_id'].unique())}')
display(pred.head(2))

# Let's find out if all of the images resulted in predictions
pred_empty = len(pred.loc[pred[pos_col].isnull(), file_col].unique())

# Filter out rows with images that do not have predictions
pred = pred.loc[~pred[pos_col].isnull()]
print(f'Total number of images in data set:             {len(dataset)}')
print(f'Images without predictions for threshold ({threshold}):{pred_empty}')
print(f'Images with predictions:                        {len(pred[file_col].unique())}')

Images in output data:     160


,image_id,image_width,image_height,batch,category_id,bbox,score,area,checkpoint,pred_threshold,file_name,ada
0,0,640,640,0,0,"[0, 74, 111, 202]",0.29196,22422,19600,0.05,test_train_219_01_05.png,1
1,105,640,640,6,14,"[377, 189, 102, 218]",0.151534,22236,19600,0.05,test_train_45_02_05.png,15


Total number of images in data set:             160
Images without predictions for threshold (0.05):0
Images with predictions:                        160


In [37]:
display(pred_raw.head())

,image_id,image_width,image_height,batch,category_id,bbox,score,area,checkpoint,pred_threshold
0,0,640,640,0,0,"[0, 74, 111, 202]",0.29196,22422,19600,0.05
1,105,640,640,6,14,"[377, 189, 102, 218]",0.151534,22236,19600,0.05
2,105,640,640,6,12,"[291, 212, 125, 219]",0.15707,27375,19600,0.05
3,105,640,640,6,13,"[291, 212, 125, 219]",0.158686,27375,19600,0.05
4,105,640,640,6,9,"[155, 253, 60, 219]",0.199029,13140,19600,0.05


### Classify predictions: TP, FP, FN ###
We create a method to classify all of the predictions in the data set. 

In [44]:
def classify_predictions(true_df, pred_df, file_col, label_col, bbox_col, score_col, iou_threshold):

    # def precision_recall(true_df, pred_df, file_col, label_col, bbox_col, score_col, iou_threshold):
    # We want to use only images with predictions
    pred_df = pred_df.loc[~pred_df[label_col].isnull()]
    
    # And the file names in the two input data frame must match
    file_list = sorted(list(set(true_df[file_col].tolist()).\
        intersection(pred_df[file_col].tolist())))
    
    classifications_df_list = []
    missed_df_list = []
    
    for f, file in enumerate(file_list):
        true_bboxes = true_df.loc[true_df[file_col] == file, bbox_col].tolist()
        pred_bboxes = pred_df.loc[pred_df[file_col] == file, bbox_col].tolist()
        
        true_bboxes = [list(np.int64(box)) for box in true_bboxes]
        pred_bboxes = [list(np.int64(box)) for box in pred_bboxes]
        
        true_labels = true_df.loc[true_df[file_col] == file, label_col].tolist()
        pred_labels = pred_df.loc[pred_df[file_col] == file, label_col].tolist()
        pred_scores = pred_df.loc[pred_df[file_col] == file, score_col].tolist()
    
        pred_cl = DetectionMetrics().\
            classify_predictions(true_labels=true_labels, 
                                 true_bboxes=true_bboxes, 
                                 pred_labels=pred_labels, 
                                 pred_bboxes=pred_bboxes, 
                                 iou_threshold=iou_threshold).\
            rename(columns={'pred_label': label_col})
    
        # Add more information about the predictions to the output
        pred_cl.insert(loc=0, column=file_col, value=file)
        pred_cl.insert(loc=1, column='iou_threshold', value=iou_threshold)
        pred_cl.insert(loc=2, column=score_col, value=pred_scores)
        pred_cl.insert(loc=3, column=bbox_col, value=pred_bboxes)
    
        classifications_df_list.append(pred_cl)
        
        # False negatives: Labels in the ground truth data that were not detected
        missed_label_list = sorted(list(set(true_labels).difference(pred_labels)))
    
        if len(missed_label_list) > 0:
            missed_cl = pd.DataFrame({label_col: missed_label_list})
            missed_cl.insert(loc=0, column=file_col, value=file)
            missed_df_list.append(missed_cl)
    
    if len(classifications_df_list) > 0:
        classifications = pd.concat(classifications_df_list, axis=0, ignore_index=True)
        classifications = classifications.\
            sort_values(by=[label_col, score_col], ascending=True).\
            reset_index(drop=True)
    else:
        classifications = None
    
    if len(missed_df_list) > 0:
        missed = pd.concat(missed_df_list, axis=0, ignore_index=True)
        missed = missed.\
            sort_values(by=label_col, ascending=True).\
            reset_index(drop=True)

    return classifications, missed

In [52]:
true_df = copy.deepcopy(df)
pred_df = copy.deepcopy(pred)
pred_df = pred_df.loc[pred_df['checkpoint'] == 19600]
bbox_col = 'bbox'
label_col = 'ada'
file_col = 'file_name'
score_col = 'score'
iou_threshold = 0.5

# Try this method
cl_data, cl_missed = classify_predictions(true_df=true_df, 
                                          pred_df=pred_df, 
                                          file_col=file_col, 
                                          label_col=label_col, 
                                          bbox_col=bbox_col, 
                                          score_col=score_col, 
                                          iou_threshold=iou_threshold)

display(cl_data.head())
display(cl_missed.head())

,file_name,iou_threshold,score,bbox,ada,TP,IoU,n_missed,duplicate_TP
0,test_train_226_14_05.png,0.5,0.050456,"[61, 245, 326, 297]",1,0,0.0,0,False
1,test_train_26_34_05.png,0.5,0.051247,"[0, 213, 103, 163]",1,0,NaN,0,False
2,test_train_87_14_05.png,0.5,0.051300,"[154, 8, 70, 108]",1,0,NaN,0,False
3,test_train_428_01_05.png,0.5,0.051367,"[152, 146, 79, 223]",1,0,NaN,0,False
4,test_train_588_01_05.png,0.5,0.051437,"[75, 210, 63, 276]",1,0,NaN,0,False


,file_name,ada
0,test_train_415_01_05.png,3
1,test_train_285_02_05.png,9


### Precision - recall for each label ###

In [67]:
def ap_from_classifications(true_df, classifications, label_col):
    """ Calculate Precision - Recall curves and AP for each label """

    pr_df_list = []
    label_list = sorted(list(classifications[label_col].unique()))
    
    for label in label_list:
    
        # Total number of positives in the data set (TP + FN)
        n_labels = len(true_df.loc[true_df[label_col] == label])
        
        # Predictions for this class sorted by score in descending order
        classifications_label = classifications.\
            loc[classifications[label_col] == label].\
            sort_values(by='score', ascending=False).\
            reset_index(drop=True)
        
        # Calculate precision and recall for each row
        correct = classifications_label['TP'].tolist()
        
        # precision = true positives / all detections
        precision=[sum(correct[:i + 1])/(i + 1) for i in range(len(correct))]
        
        # recall = true positives / samples with this label in ground truth data
        recall=[sum(correct[:i + 1])/n_labels for i in range(len(correct))]
        
        # Add precision and recall to the data frame for this label
        classifications_label = classifications_label.\
            assign(precision=precision, recall=recall)
        
        # Calculate precision and recall independent from the bounding box
        # We count every prediction that is in the image as positive
        # Detections that were not in the image did not get an iou value (FP)
        
        # TP + FP
        n_detections = len(classifications_label)
        # TP: all detections for that class with a ground truth label, so IoU >= 0
        n_detections_with_iou = len(classifications_label.loc[~classifications_label['IoU'].isnull()])
        # We can add a precision and recall value that is just for this class, indepdendent from the bounding box
        precision_label = n_detections_with_iou / n_detections
        recall_label = n_detections_with_iou / n_labels
    
        # Calculate the AUC
        auc = metrics.auc(x=recall, y=precision)
        
        # Add the class-level precision/recall values to the data frame
        classifications_label = classifications_label.\
            assign(precision_label=precision_label, 
                   recall_label=recall_label,
                   auc=auc)
        
        pr_df_list.append(classifications_label)
    
    pr_df = pd.concat(pr_df_list, axis=0, ignore_index=True)

    return pr_df

# Calculte PR curves with this function
prdf = ap_from_classifications(true_df=true_df, 
                                classifications=cl_data, 
                                label_col=label_col)

# Get the AUC values
aucdf = prdf[[label_col, 'iou_threshold', 'auc']].\
    drop_duplicates().\
    sort_values(by=label_col, ascending=True).\
    reset_index(drop=True)
aucdf.insert(loc=0, column='model', value=model_name)
aucdf.insert(loc=1, column='checkpoint', value=checkpoint)
aucdf.insert(loc=2, column='score_threshold', value=threshold)

print(f'mAP at IoU({iou_threshold}): {aucdf['auc'].mean(): .3f}')

mAP at IoU(0.5):  0.655
